# Phase 1A — Preprocessing Pipeline

**Pipeline:**
```
Input photo
  → Background removal (SegFormer) → white fill
  → Aspect-ratio resize + white padding → 512×512
  → Colour-aware quantization (32 colours, skin/lip protection)
  → Bilateral filter
  → Canny edge detection
```

**Output files to download (needed for Phase 1B):**
- `quantized.png` — 32-colour portrait (colour palette source)
- `canny_final.png` — edge map (ControlNet input)

**Run order:** Cell 0 → Restart Runtime → Cell 1 → Cell 2 → Cell 3 → Cell 4 → Cell 5 → Cell 6 → Cell 7

In [ ]:
# Cell 0: Install
# Run once, then Runtime > Restart Runtime.
import subprocess, sys

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print('[WARN]', r.stderr[-400:])

print('[1/4] core libraries')
run([sys.executable, '-m', 'pip', 'install', '--quiet',
     'opencv-python-headless', 'Pillow', 'scikit-learn', 'matplotlib'])

print('[2/4] transformers + torch (SegFormer background removal)')
run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps',
     'transformers==4.39.3', 'tokenizers==0.15.2', 'safetensors==0.4.3',
     'huggingface_hub==0.21.4'])
run([sys.executable, '-m', 'pip', 'install', '--quiet',
     'torch==2.2.1', 'torchvision==0.17.1',
     '--index-url', 'https://download.pytorch.org/whl/cu121'])

print('[3/4] accelerate')
run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', 'accelerate==0.29.3'])

print('[4/4] numpy==1.26.4 (pin last)')
run([sys.executable, '-m', 'pip', 'install', '--quiet', '--force-reinstall', 'numpy==1.26.4'])

import numpy
print(f'numpy: {numpy.__version__}  (must be 1.26.x)')
print('Done. --> Runtime > Restart Runtime, then run Cell 1 onwards.')

[1/4] core libraries
[2/4] transformers + torch (SegFormer background removal)
[3/4] accelerate
[4/4] numpy==1.26.4 (pin last)
numpy: 2.0.2  (must be 1.26.x)
Done. --> Runtime > Restart Runtime, then run Cell 1 onwards.


In [ ]:
# Cell 1: Upload photo
from google.colab import files
from PIL import Image
import io

print('Upload a portrait photo (any aspect ratio, any background)...')
uploaded = files.upload()
fname    = list(uploaded.keys())[0]
raw_img  = Image.open(io.BytesIO(uploaded[fname])).convert('RGB')
raw_img.save('/content/input_original.png')
print(f'Loaded: {fname}  size={raw_img.size}')
display(raw_img)

Upload a portrait photo (any aspect ratio, any background)...


Saving 證件照.jpg to 證件照.jpg
Loaded: 證件照.jpg  size=(1100, 1539)


In [ ]:
# Cell 2: Background removal → white fill
# Uses SegFormer (clothes segmentation model) — no scipy dependency.
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
from PIL import Image
import torch
import numpy as np

print('Loading SegFormer model (~90 MB on first run)...')
processor = SegformerImageProcessor.from_pretrained('mattmdjaga/segformer_b2_clothes')
model     = SegformerForSemanticSegmentation.from_pretrained('mattmdjaga/segformer_b2_clothes')
model.eval()

raw_img = Image.open('/content/input_original.png').convert('RGB')
inputs  = processor(images=raw_img, return_tensors='pt')
with torch.no_grad():
    logits = model(**inputs).logits

upsampled = torch.nn.functional.interpolate(
    logits, size=raw_img.size[::-1], mode='bilinear', align_corners=False)
seg_map = upsampled.argmax(dim=1).squeeze().numpy()

# Class 0 = background; keep everything else
keep_mask = (seg_map != 0).astype(np.uint8) * 255
arr       = np.array(raw_img)
white     = np.ones_like(arr) * 255
mask_3ch  = np.stack([keep_mask, keep_mask, keep_mask], axis=2)
result    = np.where(mask_3ch > 0, arr, white).astype(np.uint8)

white_bg = Image.fromarray(result)
white_bg.save('/content/bg_removed.png')
print(f'Background removed. Size: {white_bg.size}')
display(white_bg)

Loading SegFormer model (~90 MB on first run)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


/usr/local/lib/python3.12/dist-packages/transformers/models/segformer/image_processing_segformer.py:103: FutureWarning: The `reduce_labels` parameter is deprecated and will be removed in a future version. Please use `do_reduce_labels` instead.
  warnings.warn(


Background removed. Size: (1100, 1539)


In [ ]:
# Cell 3: Aspect-ratio-preserving resize + white padding → 512×512
# Scales longest side to 512, pads shorter side with white.
# Preserves full portrait without distortion or cropping.
from PIL import Image

TARGET = 512
img    = Image.open('/content/bg_removed.png').convert('RGB')
w, h   = img.size
scale  = TARGET / max(w, h)
new_w, new_h = round(w * scale), round(h * scale)
img    = img.resize((new_w, new_h), Image.LANCZOS)

canvas   = Image.new('RGB', (TARGET, TARGET), (255, 255, 255))
offset_x = (TARGET - new_w) // 2
offset_y = (TARGET - new_h) // 2
canvas.paste(img, (offset_x, offset_y))
canvas.save('/content/portrait_512.png')

print(f'Original: {w}×{h} → resized: {new_w}×{new_h} → padded: 512×512')
display(canvas)

Original: 1100×1539 → resized: 366×512 → padded: 512×512


In [ ]:
# Cell 4: Colour-aware quantization (32 colours, skin/lip protection)
# Oversamples skin and lip pixels before K-Means so they get
# dedicated cluster centroids instead of being merged into hair/clothing.
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from PIL import Image

N_COLORS   = 32
OVERSAMPLE = 16  # repeat protected pixels N times before clustering

img  = Image.open('/content/portrait_512.png').convert('RGB')
arr  = np.array(img, dtype=np.float32)
flat = arr.reshape(-1, 3)
R, G, B = flat[:,0], flat[:,1], flat[:,2]

skin_mask = (R>95)&(G>40)&(B>20)&(R>G)&(R>B)&(np.abs(R-G)>15)
lip_mask  = (R>120)&(G<110)&(B<110)&(R>G*1.3)
protect   = skin_mask | lip_mask
print(f'Protected: skin={skin_mask.sum():,}  lip={lip_mask.sum():,}  total={protect.sum():,}')

train = np.vstack([flat] + [flat[protect]] * OVERSAMPLE)
kmeans = MiniBatchKMeans(n_clusters=N_COLORS, random_state=42, n_init=5)
kmeans.fit(train)

labels    = kmeans.predict(flat)
palette   = kmeans.cluster_centers_.clip(0,255).astype(np.uint8)
quantized = palette[labels].reshape(512,512,3)

img_q = Image.fromarray(quantized)
img_q.save('/content/quantized.png')
print(f'Quantized to {N_COLORS} colours.')
display(img_q)

Protected: skin=30,693  lip=998  total=30,693
Quantized to 32 colours.


In [ ]:
# Cell 5: Bilateral filter + Canny edge detection
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Bilateral filter: smooths uniform regions, preserves edges
bgr      = cv2.imread('/content/quantized.png')
filtered = cv2.bilateralFilter(bgr, 9, 75, 75)
cv2.imwrite('/content/filtered.png', filtered)

# Canny: compare three threshold pairs, save medium as default
gray  = cv2.cvtColor(filtered, cv2.COLOR_BGR2GRAY)
pairs = [(50,150,'Low'), (100,200,'Medium (default)'), (150,300,'High')]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
best = None
for i, (lo, hi, label) in enumerate(pairs):
    edges = cv2.Canny(gray, lo, hi)
    axes[i].imshow(edges, cmap='gray')
    axes[i].set_title(f'{label}  (lo={lo}, hi={hi})')
    axes[i].axis('off')
    if i == 1: best = edges
plt.suptitle('Canny threshold comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/canny_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

# Save as 3-channel (required by ControlNet)
canny_3ch = np.stack([best, best, best], axis=2)
Image.fromarray(canny_3ch).save('/content/canny_final.png')
print('Saved: /content/canny_final.png  →  ready for Phase 1B.')

Saved: /content/canny_final.png  →  ready for Phase 1B.


In [ ]:
# Cell 6: Full pipeline summary
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

stages = [
    ('/content/input_original.png', 'Input (original)'),
    ('/content/bg_removed.png',     'Background removed'),
    ('/content/portrait_512.png',   '512×512 padded'),
    ('/content/quantized.png',      '32-colour quantized'),
    ('/content/filtered.png',       'Bilateral filtered'),
    ('/content/canny_final.png',    'Canny edges'),
]
fig, axes = plt.subplots(1, 6, figsize=(26, 5))
for ax, (path, title) in zip(axes, stages):
    ax.imshow(mpimg.imread(path), cmap='gray' if 'canny' in path else None)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.suptitle('Phase 1A — Full Preprocessing Pipeline', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/pipeline_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Phase 1A complete.')

Phase 1A complete.


In [ ]:
# Cell 7: Download outputs for Phase 1B
from google.colab import files
import os

# These two files are required inputs for Phase 1B
for path in ['/content/quantized.png', '/content/canny_final.png',
             '/content/pipeline_summary.png']:
    if os.path.exists(path):
        files.download(path)
    else:
        print(f'Missing: {path}')

print('Upload quantized.png and canny_final.png when running Phase 1B.')

Upload quantized.png and canny_final.png when running Phase 1B.
